# S4_04 — BM25 Lexical Search

> **Skilljar source**: Lesson **L06 — BM25 lexical search**
> **Week_05.md mapping**: §2.1 "BM25 Lexical Search"
> **Original file**: `_originals_skilljar/004_bm25.ipynb`

## Why a Second Search Method?

Semantic search (S4_03) is powerful when meaning matters, but it **fails on rare literal tokens**:

> Query: *"What happened with INC-2023-Q4-011?"*
>
> Semantic search returns the *Financial Analysis* section (concept of "Q4 incident" is vaguely related) instead of the *Cybersecurity* section where the exact identifier appears.

**BM25** is a classic lexical algorithm that scores documents by **token frequency × inverse document frequency × length-normalization** — the exact recipe for finding documents that literally contain your query tokens.

## What You Will Learn

1. The BM25 formula (Week_05.md §2.1.3) — `idf × tf × (k1+1) / (tf + k1·(1-b+b·|d|/avgdl))`.
2. How to build a `BM25Index` class that **mirrors the `VectorIndex` API** (`add_document`, `search`) — this uniform interface is the key to **S4_05's** hybrid retriever.
3. Tokenization matters — case-folding + non-word splitting for robustness.

## ✅ This Notebook Is Complete

Code cells are filled in from Week_05.md §2.1.4. For the blank practice version see `_student_practice_20260420/S4_04_bm25.ipynb`.

## Connection to Week_05.md

| Week_05.md Section | This Notebook |
|---|---|
| §2.1.1 "Semantic search fails on rare tokens" | Motivating example in this header |
| §2.1.2 "Tokenization" | Cell 4 — `_default_tokenizer` inside `BM25Index` |
| §2.1.3 "BM25 formula + IDF" | Cell 4 — `_compute_bm25_score` + `_calculate_idf` |
| §2.1.4 L932 "Chunk the text" | Cell 8 — `chunks = chunk_by_section(text)` |
| §2.1.4 L935-937 "Build index" | Cell 10 — `for chunk: store.add_document(...)` |
| §2.1.4 L940-945 "Search + print" | Cell 12 — `store.search("What happened with INC-2023-Q4-011?", 3)` |

## Reuse · `chunk_by_section`

Same function as S4_01 / S4_03.

In [8]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

## Core Component · `BM25Index` Class

Mirrors the API of `VectorIndex` from S4_03 (`add_document`, `search`) but with completely different internals — **no embeddings, no neural network**, just token statistics.

### BM25 hyperparameters
- `k1=1.5` — term-frequency saturation. Higher k1 ⇒ more weight to repeated terms.
- `b=0.75` — length normalization strength. `b=0` ignores document length; `b=1` fully normalizes.

> [!finding] Week_05.md §2.1.3 — Why these defaults?
> `k1=1.5`, `b=0.75` are the **Okapi BM25 reference values** used in information-retrieval research since 1994. They work well on most English corpora without tuning.

### Score normalization
The last step in `search` converts raw BM25 scores into `exp(-0.1 × score)` ∈ (0, 1]. This brings BM25 scores onto a roughly comparable scale with cosine distances — useful when you mix indexes in S4_05.

> [!tip] Tokenizer customization
> You can inject your own tokenizer (e.g., a Korean morpheme analyzer like `konlpy.Okt`) via the `tokenizer=` arg. The default lowercases + splits on non-word characters — fine for English technical text.

In [9]:
# BM25Index implementation
import math
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def _compute_bm25_score(
        self, query_tokens: List[str], doc_index: int
    ) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query_text: str,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

## Step 1 · Load Source Document

Same `report.md`. Containing the embedded `INC-2023-Q4-011` identifier that the next step will query against.

In [10]:
with open("./report.md", "r") as f:
    text = f.read()

## Step 2 · Chunk the Text

Same `chunk_by_section` we imported in cell 2. Each chunk is the body of one `## `-prefixed section.

**Week_05.md §2.1.4 reference**: line 932.

In [ ]:
# Step 1: Chunk the text by section  (Week_05.md §2.1.4 line 932)
chunks = chunk_by_section(text)

## Step 3 · Build the BM25 Index

Create a `BM25Index` and add each chunk one by one via `add_document(...)`. BM25 needs to see the whole corpus to compute IDF weights (inverse document frequency), so we add all chunks *before* searching — the first `search()` call lazily triggers `_build_index()` which computes IDF across all docs.

**Week_05.md §2.1.4 reference**: lines 935–937.

> [!tip] Why `{"content": chunk}` and not just `chunk`?
> The API matches `VectorIndex` exactly — `{"content": text}` is the document schema shared across all `SearchIndex` implementations. This uniformity is what lets S4_05's `Retriever` treat both indexes identically.

In [ ]:
# Step 2: Create a BM25 store and add each chunk  (Week_05.md §2.1.4 lines 935-937)
store = BM25Index()

for chunk in chunks:
    store.add_document({"content": chunk})

## Step 4 · Run the Canonical Query — Semantic Search's Blind Spot

Query the exact incident identifier `INC-2023-Q4-011`. Week_05.md §2.1 motivates this query: semantic search (S4_03) misranks it because the identifier is a *rare literal token* outside the embedding model's training distribution.

**Week_05.md §2.1.4 reference**: lines 940–945.

> [!finding] Expected result
> Cybersecurity (Section 10) — the section that literally contains `INC-2023-Q4-011` — should rank **first**. Software Engineering (Section 2) may rank second because the incident is cross-referenced there. This reverses the mistake semantic search made in S4_03.

In [ ]:
# Step 3: Search the store  (Week_05.md §2.1.4 lines 940-945)
results = store.search("What happened with INC-2023-Q4-011?", 3)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

## Wrap-up

BM25 is complementary to semantic search, not a replacement. **S4_05** combines both through Reciprocal Rank Fusion (RRF) — the production-grade hybrid retrieval pattern.

> [!ref] Skilljar L06 — BM25 lexical search
> Week_05.md §2.1 "BM25 Lexical Search"